# Stage 3 – Sentiment model training
**Kenyan customer-support sentiment (4 classes: Positive / Negative / Neutral / Mixed)**

Plan: (1) leakage guard, (2) TF-IDF baselines on CPU, (3) optional multilingual transformer **only if a GPU is attached**, (4) pick the best model on **validation**, (5) evaluate on **test once**, with breakdowns by language, sarcasm, negation, slang and previously-seen text.

**Input features:** `customer_inquiry` (+ optionally `conversation_context`). Label-derived columns (`sentiment_score`, `emotion`, `emotional_intensity`, `contextual_sentiment`, `sentiment_reason`, `annotation_confidence`) are deliberately **excluded**: they would not exist at prediction time and would leak the answer. `agent_reply` is excluded for the same reason.

In [ ]:
import os, re, json, time, warnings, joblib
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 120)

DATA_PATH = "/kaggle/input/datasets/linzkibet/kenya-sentiment-analysis-dataset/kenyan_customer_support_sentiment_dataset.csv"
OUT = "/kaggle/working"
TEXT, CTX, LABEL, SPLIT = "customer_inquiry", "conversation_context", "sentiment", "data_split"
LABELS = ["Negative", "Neutral", "Positive", "Mixed"]
label2id = {l: i for i, l in enumerate(LABELS)}

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except Exception:
    HAS_GPU = False
RUN_TRANSFORMER = HAS_GPU          # set True to force on CPU (very slow: expect hours)
TRANSFORMER_CANDIDATES = ["Davlan/afro-xlmr-base", "xlm-roberta-base"]   # tried in order; needs Internet ON in Kaggle settings
print("GPU available:", HAS_GPU, "| run transformer:", RUN_TRANSFORMER)

df = pd.read_csv(DATA_PATH)
df[LABEL] = df[LABEL].astype(str).str.strip().str.title()
assert set(df[LABEL]) == set(LABELS), set(df[LABEL])
df[CTX] = df[CTX].fillna("").astype(str)
df[TEXT] = df[TEXT].fillna("").astype(str)
df["y"] = df[LABEL].map(label2id)
for c in ["contains_sarcasm","contains_negation","contains_slang","contains_ambiguity","contains_context_dependency"]:
    df[c] = df[c].astype(str).str.strip().str.lower().isin(["true","1","yes","1.0"])

norm = lambda s: re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", s.lower())).strip()
df["_norm"] = df[TEXT].map(norm)
tr = df[df[SPLIT] == "Train"].reset_index(drop=True)
va = df[df[SPLIT] == "Validation"].reset_index(drop=True)
te = df[df[SPLIT] == "Test"].reset_index(drop=True)
print({k: len(v) for k, v in [("train", tr), ("val", va), ("test", te)]})

## 1. Leakage guard
Synthetic data is often templated. If val/test texts also appear in train, scores will be inflated. We flag those rows (`seen_text`) and report performance on **unseen text** separately; that is the honest number.

In [ ]:
train_texts = set(tr["_norm"])
for d in (va, te):
    d["seen_text"] = d["_norm"].isin(train_texts)
print(f"Validation texts also in train: {va['seen_text'].mean():.1%}")
print(f"Test texts also in train:       {te['seen_text'].mean():.1%}")
print(f"Unique normalised texts in train: {tr['_norm'].nunique()} / {len(tr)}")
conf = tr.groupby("_norm")[LABEL].nunique()
print("Train texts with conflicting labels:", int((conf > 1).sum()))
if va["seen_text"].mean() > .2:
    print("⚠ High overlap – read the 'unseen text' scores below, not the headline number.")

## 2. Evaluation helpers

In [ ]:
BREAK_COLS = ["language", "language_variant", "contains_sarcasm", "contains_negation", "contains_slang",
              "contains_ambiguity", "contains_context_dependency", "seen_text"]

def evaluate(name, y_true, y_pred, show=True):
    mf1, acc = f1_score(y_true, y_pred, average="macro"), accuracy_score(y_true, y_pred)
    if show:
        print(f"\n{name}: macro-F1 = {mf1:.4f} | accuracy = {acc:.4f}")
        print(classification_report(y_true, y_pred, target_names=LABELS, digits=3))
        cm = confusion_matrix(y_true, y_pred, normalize="true") * 100
        plt.figure(figsize=(5.5, 4.5)); sns.heatmap(cm, annot=True, fmt=".1f", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
        plt.xlabel("predicted"); plt.ylabel("true"); plt.title(f"{name} – row-normalised confusion (%)"); plt.tight_layout(); plt.show()
    return {"name": name, "macro_f1": mf1, "accuracy": acc}

def breakdown(y_true, y_pred, meta, cols=BREAK_COLS):
    rows = []
    for c in cols:
        if c not in meta.columns: continue
        for v, idx in meta.groupby(c).indices.items():
            if len(idx) < 30: continue
            rows.append({"slice": c, "value": v, "n": len(idx),
                         "macro_F1": f1_score(y_true[idx], y_pred[idx], average="macro"),
                         "accuracy": accuracy_score(y_true[idx], y_pred[idx])})
    return pd.DataFrame(rows).round(3)

def build_text(d, use_ctx):
    return (d[TEXT] + " [SEP] " + d[CTX]).values if use_ctx else d[TEXT].values

## 3. TF-IDF baselines (CPU)
Word 1–2-grams + character 2–5-grams (robust to Sheng spellings, typos and code-switching). Four variants: {LogReg, LinearSVC} × {inquiry only, inquiry + context}.

In [ ]:
def make_pipe(clf):
    feats = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=300_000)),
    ])
    return Pipeline([("tfidf", feats), ("clf", clf)])

variants = {
    "LogReg | inquiry":          (lambda: make_pipe(LogisticRegression(C=4, max_iter=2000, random_state=SEED)), False),
    "LinearSVC | inquiry":       (lambda: make_pipe(LinearSVC(C=0.5, random_state=SEED)), False),
    "LogReg | inquiry+context":  (lambda: make_pipe(LogisticRegression(C=4, max_iter=2000, random_state=SEED)), True),
    "LinearSVC | inquiry+context": (lambda: make_pipe(LinearSVC(C=0.5, random_state=SEED)), True),
}
candidates, results = {}, []
for name, (factory, use_ctx) in variants.items():
    t0 = time.time()
    pipe = factory().fit(build_text(tr, use_ctx), tr["y"])
    pv = pipe.predict(build_text(va, use_ctx))
    r = evaluate(name, va["y"].values, pv, show=False)
    unseen = ~va["seen_text"].values
    r["macro_f1_unseen_text"] = f1_score(va["y"].values[unseen], pv[unseen], average="macro") if unseen.sum() > 30 else np.nan
    r["fit_seconds"] = round(time.time() - t0, 1)
    results.append(r)
    candidates[name] = {"val_pred": pv, "model": pipe, "use_ctx": use_ctx,
                        "predict": (lambda d, p=pipe, u=use_ctx: p.predict(build_text(d, u)))}
res = pd.DataFrame(results).set_index("name").round(4)
display(res.sort_values("macro_f1", ascending=False))

## 4. Transformer fine-tuning (GPU only)
On Kaggle: **Settings → Accelerator → GPU T4 ×2** and **Internet → On**, then re-run from the top. On CPU this cell is skipped.

In [ ]:
if RUN_TRANSFORMER:
    import torch, inspect
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                              Trainer, DataCollatorWithPadding, set_seed)
    set_seed(SEED)
    tok = model = None
    for mname in TRANSFORMER_CANDIDATES:
        try:
            tok = AutoTokenizer.from_pretrained(mname)
            model = AutoModelForSequenceClassification.from_pretrained(
                mname, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=label2id)
            print("Loaded", mname); break
        except Exception as e:
            print(f"Could not load {mname}: {type(e).__name__}: {str(e)[:120]}")
    if model is None:
        print("⚠ No transformer could be loaded (is Internet enabled?). Skipping.")
    else:
        MAXLEN = 128   # p99 of inquiry is ~40 words; leaves room for context as second segment
        class DS(torch.utils.data.Dataset):
            def __init__(self, d):
                enc = tok(list(d[TEXT]), list(d[CTX]) if USE_CTX_TF else None, truncation=True, max_length=MAXLEN)
                self.enc, self.y = enc, d["y"].tolist()
            def __len__(self): return len(self.y)
            def __getitem__(self, i):
                item = {k: v[i] for k, v in self.enc.items()}; item["labels"] = self.y[i]; return item
        USE_CTX_TF = True   # contextual sentiment is part of the task; set False to use the inquiry only
        ds_tr, ds_va, ds_te = DS(tr), DS(va), DS(te)
        kw = dict(output_dir=f"{OUT}/transformer", num_train_epochs=3, per_device_train_batch_size=32,
                  per_device_eval_batch_size=64, learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
                  save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="macro_f1",
                  greater_is_better=True, fp16=True, report_to="none", seed=SEED, save_total_limit=1)
        key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"
        kw[key] = "epoch"
        def cm(p): 
            return {"macro_f1": f1_score(p.label_ids, p.predictions.argmax(1), average="macro"),
                    "accuracy": accuracy_score(p.label_ids, p.predictions.argmax(1))}
        trainer = Trainer(model=model, args=TrainingArguments(**kw), train_dataset=ds_tr, eval_dataset=ds_va,
                          data_collator=DataCollatorWithPadding(tok), compute_metrics=cm)
        trainer.train()
        pv = trainer.predict(ds_va).predictions.argmax(1)
        name = f"Transformer | {model.config._name_or_path}"
        r = evaluate(name, va["y"].values, pv, show=False)
        unseen = ~va["seen_text"].values
        r["macro_f1_unseen_text"] = f1_score(va["y"].values[unseen], pv[unseen], average="macro") if unseen.sum() > 30 else np.nan
        results.append(r)
        candidates[name] = {"val_pred": pv, "model": trainer, "use_ctx": USE_CTX_TF,
                            "predict": lambda d, t=trainer: t.predict(DS(d)).predictions.argmax(1)}
        display(pd.DataFrame(results).set_index("name").round(4).sort_values("macro_f1", ascending=False))
else:
    print("Skipped (no GPU). The best TF-IDF model below is your CPU baseline.")

## 5. Select the best model on validation and diagnose it
Selection uses **validation macro-F1 only**. The test set is untouched until the next section.

In [ ]:
best_name = max(candidates, key=lambda k: f1_score(va["y"].values, candidates[k]["val_pred"], average="macro"))
print("Selected:", best_name)
pv = candidates[best_name]["val_pred"]
evaluate(best_name + " (validation)", va["y"].values, pv)
print("\nValidation performance by slice (look for weak spots):")
display(breakdown(va["y"].values, pv, va))

In [ ]:
wrong = va[pv != va["y"].values].copy()
wrong["pred"] = [LABELS[i] for i in pv[pv != va["y"].values]]
print(f"{len(wrong)} validation errors. Sample:")
display(wrong[[TEXT, LABEL, "pred", "language", "contains_sarcasm", "contains_negation"]].sample(min(15, len(wrong)), random_state=1))
print("Most common confusions (true → predicted):")
print(wrong.groupby([LABEL, "pred"]).size().sort_values(ascending=False).head(8).to_string())

## 6. Final test evaluation (run once)

In [ ]:
pt = candidates[best_name]["predict"](te)
final = evaluate(best_name + " (TEST)", te["y"].values, pt)
bd = breakdown(te["y"].values, pt, te)
display(bd)
unseen = ~te["seen_text"].values
if unseen.sum() > 30:
    print(f"\nTest macro-F1 on UNSEEN text only (n={unseen.sum()}): {f1_score(te['y'].values[unseen], pt[unseen], average='macro'):.4f}")

## 7. Save artefacts

In [ ]:
os.makedirs(OUT, exist_ok=True)
m = candidates[best_name]["model"]
if hasattr(m, "predict") and not hasattr(m, "train"):
    joblib.dump(m, f"{OUT}/best_sentiment_model.joblib"); print("Saved best_sentiment_model.joblib")
else:
    m.save_model(f"{OUT}/best_transformer"); tok.save_pretrained(f"{OUT}/best_transformer"); print("Saved best_transformer/")
pd.DataFrame({"ticket_id": te["ticket_id"], "true": te[LABEL], "pred": [LABELS[i] for i in pt]}).to_csv(f"{OUT}/test_predictions.csv", index=False)
bd.to_csv(f"{OUT}/test_breakdown.csv", index=False)
json.dump({"selected_model": best_name, "validation": pd.DataFrame(results).round(4).to_dict("records"),
           "test_macro_f1": round(final["macro_f1"], 4), "test_accuracy": round(final["accuracy"], 4)},
          open(f"{OUT}/stage_3_metrics.json", "w"), indent=2)
print(json.dumps({"selected_model": best_name, "test_macro_f1": round(final["macro_f1"], 4)}, indent=2))

## How to read the results
* **Headline vs unseen-text score:** if they differ a lot, the model is memorising templates; trust the unseen number.
* **Sarcasm / negation / ambiguity slices:** bag-of-words models usually fail here; this is where a transformer should help.
* **Mixed ↔ Neutral confusion** is the most common error for 4-class sentiment and is partly a labelling-definition problem, not only a model problem.
* Near-perfect scores (>0.98) on synthetic data mean the task is easy by construction. Validate on a small set of **real** Kenyan customer messages before trusting it.